# 18 — Paper figure F + G replica using OUR data

Replicates the layout of CellOT paper (Bunne et al., 2023) panels f and g but with our 5-flavor matrix data and our models (scGen, IMPACT_CellOT).

**Reusable**: the bar-plot and density-overlay functions live in `scripts/render_results_figures.py` so that:
- This notebook calls them with our 80-cell matrix as data source.
- Future notebooks (e.g. when BCG-vs-real-human ground truth becomes available) can call the same functions with that data source — no code rewrite needed.

## Figure F replica
- One subplot per flavor (5 subplots), x-axis = group (A/B/C/D), bars colored by model (scGen vs IMPACT_CellOT). Y-axis: R² of means (top panel) and MMD (bottom panel). Mirrors paper's two stacked panels.

## Figure G replica
- For one chosen `(flavor, group, mode)` matrix entry, density-overlay of the curated 12-marker panel showing actual mouse / actual human / scGen prediction / IMPACT prediction. Same layout as paper's 4-panel grid but with our markers.

In [1]:
import os, sys, warnings
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import anndata as ad

warnings.filterwarnings("ignore")
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts")
import render_results_figures as rrf

BASE = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
RESULTS = BASE / "cellot/cellot_gpu/results"
OUT_DIR = BASE / "speciesOT/baseline/analysis/paper_figure_replica_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FLAVORS = ["seurat", "cell_ranger", "seurat_v3", "seurat_v3_paper", "pearson_residuals"]
GROUPS = ["a","b","c","d"]
MODES = ["ood","iid"]
MODELS = ["scgen", "impact_cellot"]

R2_METRICS = {"r2-means", "r2-stds", "r2-pairwise_feat_corrs"}

# Same-as-notebook-13 long-form gather, with the r->R² squaring fix
def gather_long(flavors=FLAVORS, where="latent_space"):
    rows = []
    for f, g, m, model in product(flavors, GROUPS, MODES, MODELS):
        p = RESULTS / f"hvg_{f}_{g}_{m}" / model / f"evals_ood_{where}" / "evals.csv"
        if not p.exists() or p.stat().st_size < 200:
            continue
        df = pd.read_csv(p)
        is_r2 = df["metric"].isin(R2_METRICS)
        df.loc[is_r2, "value"] = df.loc[is_r2, "value"] ** 2
        df["flavor"] = f; df["group"] = g; df["mode"] = m; df["model"] = model
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

print("setup OK")
print("rrf module:", rrf.__file__)

setup OK
rrf module: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/render_results_figures.py


## 1. Figure F replica — bar plots of R² and MMD per (flavor, group, model), faceted by mode

In [2]:
long_latent = gather_long(where="latent_space")
print(f"Loaded {len(long_latent)} rows from latent-space evals")
print("group values:", sorted(long_latent["group"].unique()))

# Figure F: R² of means, faceted by flavor, x=group, hue=model. One per mode.
for mode in MODES:
    sub = long_latent[long_latent["mode"] == mode]
    rrf.plot_metric_bars(
        sub,
        metric="r2-means",
        group_by="group",
        hue_by="model",
        facet_by="flavor",
        facet_order=FLAVORS,
        group_order=GROUPS,        # lowercase a/b/c/d matching the data
        hue_order=MODELS,
        hue_palette={"scgen": "#bbbbbb", "impact_cellot": "#e74c3c"},
        metric_label=f"R² of means ({mode.upper()}, latent space)",
        ylim=(0, 1),
        out_path=FIG_DIR / f"figure_F_R2_{mode.upper()}",
    )

for mode in MODES:
    sub = long_latent[long_latent["mode"] == mode]
    rrf.plot_metric_bars(
        sub,
        metric="mmd",
        group_by="group",
        hue_by="model",
        facet_by="flavor",
        facet_order=FLAVORS,
        group_order=GROUPS,
        hue_order=MODELS,
        hue_palette={"scgen": "#bbbbbb", "impact_cellot": "#e74c3c"},
        metric_label=f"MMD ({mode.upper()}, latent space)  ↓ lower is better",
        out_path=FIG_DIR / f"figure_F_MMD_{mode.upper()}",
    )

print("\nFigure F replicas saved (4 figures: R²/MMD x OOD/IID).")

Loaded 17280 rows from latent-space evals
group values: ['a', 'b', 'c', 'd']


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_R2_OOD.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_R2_OOD.png


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_R2_IID.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_R2_IID.png


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_MMD_OOD.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_MMD_OOD.png


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_MMD_IID.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_F_MMD_IID.png

Figure F replicas saved (4 figures: R²/MMD x OOD/IID).


## 2. Figure G replica — biomarker density for one chosen cell

Picks the best IMPACT cell by latent R² (across all 80 matrix entries). Loads the per-flavor source dataset (gene space) and the per-model `imputed.h5ad` from the data-space evaluation. Calls `plot_marginals_paper_style()`.

In [3]:
DATA_DIR = BASE / "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg"

MARKER_PANEL = {
    "PTPRC (CD45)":  "ENSG00000081237",
    "CD3E":          "ENSG00000198851",
    "CD4":           "ENSG00000010610",
    "CD8A":          "ENSG00000153563",
    "CD5":           "ENSG00000110448",
    "CD7":           "ENSG00000173762",
    "CCR7":          "ENSG00000126353",
    "NCAM1 (CD56)":  "ENSG00000149294",
    "MS4A1 (CD20)":  "ENSG00000156738",
    "CD19":          "ENSG00000177455",
    "CD14":          "ENSG00000170458",
    "ITGAM (CD11b)": "ENSG00000169896",
}

HOLDOUT_LUT = {
    "a": ["CL:0000625"],
    "b": ["CL:0000625", "CL:0000893"],
    "c": ["CL:0000624", "CL:0000625", "CL:0000893"],
    "d": ["CL:0000624"],
}

# Pick best IMPACT cell by latent R^2 of means (true R²)
sub = long_latent[(long_latent["model"] == "impact_cellot") & (long_latent["metric"] == "r2-means")]
agg = sub.groupby(["flavor", "group", "mode", "ncells"], as_index=False)["value"].mean()
largest = agg.sort_values("ncells").drop_duplicates(["flavor","group","mode"], keep="last")
best = largest.sort_values("value", ascending=False).iloc[0]
best_flavor, best_gk, best_mode = best["flavor"], best["group"], best["mode"]
print(f"Best IMPACT cell: {best_flavor} / Group {best_gk.upper()} / {best_mode.upper()}  R²={best['value']:.4f}")


def to_dense(X):
    return np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)


def render_figure_G(flavor, gk, mode, out_stem):
    src_path = DATA_DIR / f"hvg_{flavor}_{gk}_v07.h5ad"
    if not src_path.exists():
        print(f"  src missing: {src_path}"); return
    src = ad.read_h5ad(src_path)
    is_holdout = src.obs["cell_type_ontology_term_id"].astype(str).isin(HOLDOUT_LUT[gk])
    actual_mouse = src[(src.obs["condition"]=="mouse") & is_holdout]
    actual_human = src[(src.obs["condition"]=="human") & is_holdout]

    # Load data-space imputed for both models. Skip if shape is 50d (latent leak)
    n_genes_atlas = src.n_vars  # the gene namespace size, e.g. 1000
    pred = {}
    for model, label in [("scgen","scGen pred"), ("impact_cellot","IMPACT pred")]:
        p = BASE / "cellot/cellot_gpu/results" / f"hvg_{flavor}_{gk}_{mode}" / model / "evals_ood_data_space" / "imputed.h5ad"
        if not p.exists():
            print(f"  skip {label}: no imputed.h5ad"); continue
        a_pred = ad.read_h5ad(p)
        if a_pred.n_vars != n_genes_atlas:
            print(f"  skip {label}: imputed has {a_pred.n_vars} dims, expected {n_genes_atlas} (probably a latent-leak; re-run eval with --embedding ae)")
            continue
        pred[label] = to_dense(a_pred.X)
    if not pred:
        print(f"  no usable data-space imputed for {flavor}/{gk}/{mode}"); return

    var_lookup = {ensg: i for i, ensg in enumerate(src.var_names.astype(str))}
    rrf.plot_marginals_paper_style(
        actual_target=to_dense(actual_human.X),
        predicted_traces=pred,
        gene_panel=MARKER_PANEL,
        var_index_lookup=var_lookup,
        actual_source=to_dense(actual_mouse.X) if len(actual_mouse) > 0 else None,
        n_cols=4,
        title=f"Biomarker density — {flavor} / Group {gk.upper()} / {mode.upper()} (best IMPACT R²={best['value']:.3f})",
        out_path=FIG_DIR / out_stem,
    )

render_figure_G(best_flavor, best_gk, best_mode, "figure_G_best_IMPACT_cell")
print("\nFigure G replica saved.")

Best IMPACT cell: seurat_v3 / Group D / OOD  R²=0.9184


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_best_IMPACT_cell.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_best_IMPACT_cell.png

Figure G replica saved.


## 3. Bonus: side-by-side Pearson vs other-flavor at the same (group, mode)

Renders Figure G for two cells back-to-back, using the same biomarker panel — easy comparison for the meeting.

In [4]:
side_by_side = [
    ("pearson_residuals", "c", "ood"),
    ("seurat",            "c", "ood"),
]
for flavor, gk, mode in side_by_side:
    out = f"figure_G_{flavor}_{gk}_{mode}"
    print(f"\nrendering {out}...")
    render_figure_G(flavor, gk, mode, out)
print("\nDone — Figure G side-by-side saved.")


rendering figure_G_pearson_residuals_c_ood...


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_pearson_residuals_c_ood.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_pearson_residuals_c_ood.png

rendering figure_G_seurat_c_ood...


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_seurat_c_ood.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/paper_figure_replica_outputs/figures/figure_G_seurat_c_ood.png

Done — Figure G side-by-side saved.
